In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast
from itertools import combinations
from sklearn.metrics import cohen_kappa_score

In [ ]:
df = pd.read_csv("map_file.csv")

# Class distribution by sex and age

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df, x='age', hue='sex', bins=15, multiple='stack')
plt.title('Age Distribution by Sex', fontsize = '20')
plt.xlabel('Age', fontsize = '20')
plt.ylabel('Frequency', fontsize = '20')
plt.legend(title='Sex', labels=['Male', 'Female'], fontsize = '15')
plt.grid(True)
plt.savefig("age_distribution.pdf", bbox_inches='tight')

# Annotator agreement

In [ ]:
original_names = ['doctor1', 'doctor2', 'doctor3', 'doctor4', 'doctor5', 'doctor6', 'doctor7']
names = ["Doctor 1", "Doctor 2", "Doctor 3", "Doctor 4", "Doctor 5", "Doctor 6", "Doctor 7"]
specific_classes = ["AFIB", "CRBBB", "IRBBB", "CLBBB", "1AVB", "PVC"]

In [ ]:
def parse_annotations(value):
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return np.nan
    return parsed

def presence_absence(annotation_list, specific_class):
    if isinstance(annotation_list, list) and len(annotation_list) > 0:
        return int(specific_class in annotation_list)
    return np.nan

df_copy = df.copy()

for col in original_names:
    df_copy[col] = df_copy[col].apply(parse_annotations)
df_copy.rename(columns=dict(zip(original_names, names)), inplace=True)

for specific_class in specific_classes:
    for annotator in names:
        df_copy[f"{annotator}_{specific_class}_pa"] = df_copy[annotator].apply(
            lambda x: presence_absence(x, specific_class)
        )

kappa_scores = {specific_class: {} for specific_class in specific_classes}
for specific_class in specific_classes:
    for annotator_1, annotator_2 in combinations(names, 2):
        col_1 = f"{annotator_1}_{specific_class}_pa"
        col_2 = f"{annotator_2}_{specific_class}_pa"
        pair_df = df_copy[[col_1, col_2]].dropna()
        if not pair_df.empty:
            kappa_scores[specific_class][(annotator_1, annotator_2)] = cohen_kappa_score(
                pair_df[col_1], pair_df[col_2]
            )

num_classes = len(specific_classes)
rows = (num_classes + 2) // 3
fig, axes = plt.subplots(rows, 3, figsize=(15, 5 * rows))
axes = axes.flatten()

for idx, specific_class in enumerate(specific_classes):
    heatmap_data = pd.DataFrame(np.nan, index=names, columns=names)
    np.fill_diagonal(heatmap_data.values, 1.0)

    for (annotator_1, annotator_2), kappa_score in kappa_scores[specific_class].items():
        heatmap_data.loc[annotator_1, annotator_2] = kappa_score
        heatmap_data.loc[annotator_2, annotator_1] = kappa_score

    sns.heatmap(heatmap_data, ax=axes[idx], annot=True, cmap="coolwarm", vmin=-1, vmax=1)
    axes[idx].set_title(f"Cohen's Kappa for class '{specific_class}'")

for ax in axes[num_classes:]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig("cohen_kappa_metrics_2_row_layout.pdf", bbox_inches="tight")
plt.show()

# Data stastistics

In [ ]:
stat_classes = ["AFIB", "CRBBB", "IRBBB", "CLBBB", "1AVB", "PVC", "AFLT", "SARRH", "WPW", "LAFB"]

In [ ]:
agreement_df = df.copy()

doctor_cols = original_names

for col in doctor_cols:
    agreement_df[col] = agreement_df[col].apply(parse_annotations)

agreement_df['unified_decision'] = agreement_df['unified_decision'].apply(parse_annotations)
segmented_mask = agreement_df.get('segmented', pd.Series(False, index=agreement_df.index)).astype(bool)

def has_abnormality(annotations, abnormality):
    return any(abnormality in item for item in annotations)

rows = []
for abnormality in stat_classes:
    presence = 0
    ambiguity = 0

    abnormality_mask = agreement_df['unified_decision'].apply(lambda values: has_abnormality(values, abnormality))
    num_segmented = int((segmented_mask & abnormality_mask).sum())

    for _, row in agreement_df.iterrows():
        reviewed_annotations = [row[col] for col in doctor_cols if len(row[col]) > 0]
        num_reviewed = len(reviewed_annotations)

        num_agree = sum(has_abnormality(annotations, abnormality) for annotations in reviewed_annotations)

        if num_agree == 0:
            continue

        if num_agree == num_reviewed:
            presence += 1
        else:
            ambiguity += 1

    rows.append({
        "Abnormality": abnormality,
        "Presence": presence,
        "Ambiguity": ambiguity,
        "Number of ECGs segmented": num_segmented,
    })

abnormality_agreement_table = pd.DataFrame(rows).set_index("Abnormality")
abnormality_agreement_table